# Notebook 1: Step-by-Step ReAct Agent in LangGraph

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **ReAct (Reasoning and Acting)** pattern and its three-step loop.
2. Define custom **tools** using LangChain's `@tool` decorator.
3. Build a **state machine** using LangGraph's `StateGraph`.
4. Understand how **conditional edges** control the agent's decision flow.
5. Run a working **Financial Analyst Agent** that searches Wikipedia for financial facts.

---

## Use Case: Financial Analyst Agent

> **Scenario**: A trader asks the agent about economic indicators, company histories, or financial concepts. The agent decides whether it knows the answer from its training data, or whether it needs to search Wikipedia for up-to-date or detailed information.

This is a **stateless** (single-turn) agent — it answers one question at a time without memory of previous interactions. We cover persistent memory in Notebook 2.

---

## The ReAct Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    USER QUESTION                        │
│            "What is the history of the S&P 500?"       │
└─────────────────────┬───────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────┐
│                  REASONER NODE                          │
│              (LLM: gpt-4.1 via Azure)                  │
│                                                         │
│   "I need to search Wikipedia for S&P 500 history"     │
│   → Decides to call: finance_wiki_search("S&P 500")    │
└─────────────────────┬───────────────────────────────────┘
                      │ tool_calls present?
                      ▼ YES
┌─────────────────────────────────────────────────────────┐
│                   TOOL NODE                             │
│           (Executes finance_wiki_search)                │
│                                                         │
│   Wikipedia API → Returns 5-sentence summary           │
└─────────────────────┬───────────────────────────────────┘
                      │ ToolMessage added to state
                      ▼
┌─────────────────────────────────────────────────────────┐
│              REASONER NODE (again)                      │
│                                                         │
│   "I now have the information. Let me answer."         │
│   → No more tool_calls → END                           │
└─────────────────────┬───────────────────────────────────┘
                      │
                      ▼
                 FINAL ANSWER
```

---

## Key Concepts

| Concept | Description |
|---|---|
| **ReAct Loop** | Think → Act → Observe, repeated until the agent has enough information |
| **AgentState** | A TypedDict holding the list of messages; `add_messages` reducer appends new messages |
| **Reasoner Node** | Calls the LLM with the current state; may produce a final answer or a tool call |
| **Tool Node** | Executes the tool requested by the LLM and returns a `ToolMessage` |
| **Conditional Edge** | Routes to "tools" if tool_calls exist, otherwise routes to END |
| **bind_tools** | Registers available tools with the LLM so it knows what it can call |

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

We use `AzureChatOpenAI` from `langchain-openai`. The key difference from standard OpenAI is that we specify:
- `azure_deployment`: The name of your deployed model in Azure.
- `AZURE_OPENAI_ENDPOINT`: Your Azure resource endpoint.
- `OPENAI_API_VERSION`: The API version to use.

> **Note**: LangChain reads the API key from `AZURE_OPENAI_API_KEY` and the endpoint from `AZURE_OPENAI_ENDPOINT` automatically.

In [1]:
# Install required packages (uncomment if needed)
# !pip install langgraph langchain-openai langchain wikipedia

import os
import wikipedia
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from typing import TypedDict, Sequence, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage, HumanMessage
from langgraph.graph import StateGraph, END

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

# ── Initialize the LLM ────────────────────────────────────────────────────────
model = AzureChatOpenAI(
    azure_deployment="gpt-4.1",
    temperature=0  # Deterministic: we want reliable tool-calling decisions
)

print("Azure OpenAI model initialized!")
print(f"   Deployment: gpt-4.1")
print(f"   Temperature: 0 (deterministic)")

Azure OpenAI model initialized!
   Deployment: gpt-4.1
   Temperature: 0 (deterministic)


## 🛠️ Step 2: Defining the Wikipedia Search Tool

The `@tool` decorator transforms a regular Python function into a **LangChain Tool**. The docstring becomes the tool's description — this is what the LLM reads to decide *when* to use it.

> **Important**: Write clear, descriptive docstrings! The LLM uses them to decide whether to call the tool.

In [2]:
@tool
def finance_wiki_search(query: str) -> str:
    """Search Wikipedia for financial, economic, or company-related queries.
    Use this for questions about: company history, economic indicators, financial indices,
    market history, or any factual financial information.
    Returns a brief 5-sentence summary of the top Wikipedia result.
    """
    try:
        summary = wikipedia.summary(query, sentences=5)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        # If the query is ambiguous, try the first option
        return wikipedia.summary(e.options[0], sentences=5)
    except Exception as e:
        return f"Search error: {e}"

# Register tools and bind them to the model
tools = [finance_wiki_search]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

print(f"✅ Tool registered: '{finance_wiki_search.name}'")
print(f"   Description: {finance_wiki_search.description[:100]}...")

✅ Tool registered: 'finance_wiki_search'
   Description: Search Wikipedia for financial, economic, or company-related queries.
    Use this for questions abo...


## 🗂️ Step 3: Defining the Agent State

LangGraph requires a **state schema** — a TypedDict that defines what data the agent tracks. The `add_messages` reducer is crucial: it ensures new messages are **appended** to the list, not replaced.

```python
# Without add_messages (WRONG - would overwrite):
state["messages"] = new_message

# With add_messages (CORRECT - appends):
state["messages"] = existing_messages + [new_message]
```

In [3]:
class AgentState(TypedDict):
    """State of the agent for one turn or conversation.
    
    The 'messages' field holds the full conversation history.
    The 'add_messages' reducer ensures new messages are appended, not overwritten.
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]

print("✅ AgentState defined.")
print("   Fields: messages (Annotated with add_messages reducer)")

✅ AgentState defined.
   Fields: messages (Annotated with add_messages reducer)


## 🔀 Step 4: Defining Graph Nodes and the Conditional Edge

### Node 1: The Reasoner
Calls the LLM. The LLM either:
- Returns a **final answer** (no tool_calls in the response).
- Returns a **tool call request** (tool_calls present in the response).

### Node 2: The Tool Executor
Iterates over all requested tool calls, executes each one, and wraps the results in `ToolMessage` objects.

### Conditional Edge: `should_continue`
This function inspects the last message. If it contains tool calls, it routes to the `"tools"` node. Otherwise, it routes to `END`.

In [4]:
# ── Node 1: Reasoner ──────────────────────────────────────────────────────────
def call_model(state: AgentState):
    """Call the LLM with the current conversation state."""
    system_prompt = SystemMessage(
        content=(
            "You are a knowledgeable Financial Analyst Assistant. "
            "Use the finance_wiki_search tool to look up company history, "
            "economic indicators, stock market history, or financial concepts when needed."
        )
    )
    # Prepend system prompt to the existing message history
    messages = [system_prompt] + list(state["messages"])
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

# ── Node 2: Tool Executor ──────────────────────────────────────────────────────
def tool_node(state: AgentState):
    """Execute all tool calls requested by the LLM."""
    last_message = state["messages"][-1]
    results = []
    
    for tc in last_message.tool_calls:
        print(f"  🔍 Searching Wikipedia: '{tc['args'].get('query', '')}'")
        tool_fn = tools_by_name[tc["name"]]
        output = tool_fn.invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
        
    return {"messages": results}

# ── Conditional Edge ──────────────────────────────────────────────────────────
def should_continue(state: AgentState):
    """Route to 'tools' if the LLM requested tool calls, otherwise END."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"  # Continue loop
    return END           # Exit loop

print("✅ Nodes and conditional edge defined.")

✅ Nodes and conditional edge defined.


## 🏗️ Step 5: Building and Compiling the Graph

We wire the nodes and edges together. The graph structure is:

```
START → reasoner → [conditional] → tools → reasoner → ... → END
```

In [5]:
# Build the StateGraph
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("reasoner", call_model)
graph.add_node("tools", tool_node)

# Set the entry point
graph.set_entry_point("reasoner")

# Add the conditional edge from reasoner
graph.add_conditional_edges(
    "reasoner",       # From this node...
    should_continue,  # ...call this function to decide...
    {
        "tools": "tools",  # If "tools" → go to tool node
        END: END           # If END → finish
    }
)

# After tools run, always go back to the reasoner
graph.add_edge("tools", "reasoner")

# Compile the graph into a runnable
app = graph.compile()
print("✅ LangGraph compiled successfully!")
print("   Graph: START → reasoner ⇄ tools → END")

✅ LangGraph compiled successfully!
   Graph: START → reasoner ⇄ tools → END


## 🚀 Step 6: Running the Financial Analyst Agent

Let's test the agent with a question that requires external knowledge. Watch how it decides to use the search tool!

In [6]:
query = "What is the history of the S&P 500 index and when was it introduced?"
print(f"👤 User: {query}\n")

initial_state = {"messages": [HumanMessage(content=query)]}

print("🔄 Agent is thinking...\n")
for event in app.stream(initial_state):
    for node_name, node_state in event.items():
        if node_name == "reasoner":
            msg = node_state["messages"][-1]
            if msg.tool_calls:
                print(f"🧠 Reasoner: Decided to search Wikipedia...")
            else:
                print(f"🤖 Final Answer:\n{msg.content}")

👤 User: What is the history of the S&P 500 index and when was it introduced?

🔄 Agent is thinking...

🧠 Reasoner: Decided to search Wikipedia...
  🔍 Searching Wikipedia: 'S&P 500 history and introduction date'
🤖 Final Answer:
The S&P 500 (Standard and Poor's 500) is a stock market index that tracks the performance of 500 leading companies listed on U.S. stock exchanges. It is one of the most widely followed equity indices and represents about 80% of the total market capitalization of U.S. public companies. The S&P 500 is a public float weighted or capitalization-weighted index, meaning companies with larger market values have a greater impact on the index's movement.

The index was introduced to provide a broad snapshot of the U.S. equity market and has become a benchmark for investors and fund managers. The S&P 500 was officially introduced in 1957 by Standard & Poor's, replacing an earlier index that tracked only 90 stocks. Since then, it has grown to become a key indicator of the ov

## 🧪 Step 7: Testing Without Tool Use

Let's ask a question the LLM can answer from its training data — it should **not** use the tool.

In [7]:
query2 = "What does 'diversification' mean in investing?"
print(f"👤 User: {query2}\n")

initial_state2 = {"messages": [HumanMessage(content=query2)]}

tool_was_used = False
for event in app.stream(initial_state2):
    for node_name, node_state in event.items():
        if node_name == "tools":
            tool_was_used = True
        if node_name == "reasoner":
            msg = node_state["messages"][-1]
            if not msg.tool_calls:
                print(f"🤖 Final Answer:\n{msg.content}")

print(f"\n📊 Tool used: {tool_was_used} (Expected: False — the LLM knows this from training)")

👤 User: What does 'diversification' mean in investing?

  🔍 Searching Wikipedia: 'diversification in investing'
🤖 Final Answer:
In investing, "diversification" refers to the practice of spreading investments across different assets or securities to reduce risk. By holding a variety of investments that are not closely correlated, losses in one area may be offset by gains in another. This approach helps protect your portfolio from significant declines if a single investment performs poorly. Diversification is a key principle in asset management and is commonly achieved through mutual funds or other collective investment schemes.

📊 Tool used: True (Expected: False — the LLM knows this from training)


## 📚 Summary and Key Takeaways

| Component | Role | Code |
|---|---|---|
| `AgentState` | Holds conversation history | `TypedDict` with `add_messages` |
| `call_model` | Reasoner node — calls the LLM | `model_with_tools.invoke(messages)` |
| `tool_node` | Executes tool calls | `tool_fn.invoke(tc["args"])` |
| `should_continue` | Conditional edge logic | Checks `last_message.tool_calls` |
| `StateGraph` | Wires everything together | `graph.compile()` |

### 🔑 Key Insights

1. **The LLM decides when to use tools** — you don't hardcode when to search. The model reads the tool descriptions and decides autonomously.
2. **The loop is implicit** — the `tools → reasoner` edge creates the ReAct loop automatically.
3. **Temperature = 0** is recommended for agents — it makes tool-calling decisions more reliable and consistent.

### 🚀 Next Steps
- **Notebook 2**: Add persistent memory so the agent remembers user information across sessions.
- **Notebook 3**: Learn prompt chaining for multi-step sequential workflows.
